## HYPERPARAMETER TUNING

This code implements **Bayesian hyperparameter optimization** for the BBTransformer using **Optuna**, targeting **F1 score** on a validation set. 

- It searches over architecture (e.g., `embed_dim`, `num_heads`), regularization (`dropout_*`), training (`lr`, `weight_decay`), and loss (`gamma` for adaptive focal loss).  
- Each trial trains a fresh model with **reproducible seeds**, **mixed-precision**, and **early stopping**, then reports validation F1 (with AUC for diagnostics).  
- The tuner uses **MedianPruner** for efficiency and returns the best configuration for final training.

In [ ]:
# Libraries
import os
import json
import argparse
import numpy as np
import pandas as pd
import bbtransformer as bbt
import matplotlib.pyplot as plt


from bbt import (
    prepare_fmri_data,
    tune_hyperparameters
)

# STEP 1: Load data
print("="*60)
print("STEP 1: Loading Data")
print("="*60)

pheno = pd.read_csv('dataset/ukbb_pheno.csv')
features = np.load('dataset/ukbb_features_zscored.npz')
data = features['data']
subject_ids = features['subject_ids']

required_columns = ['Sex', 'Age', 'neuro_healthy']
missing = [col for col in required_columns if col not in pheno.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# STEP 2: Prepare data loaders
print("\n" + "="*60)
print("STEP 2: Preparing Data Loaders")
print("="*60)

train_loader, val_loader, _, metadata = prepare_fmri_data(
    data_path='dataset/ukbb_features_zscored.npz',
    pheno_path='dataset/ukbb_pheno.csv',
    target_column='Sex',
    age_column='Age',
    ext_column='neuro_healthy',
    batch_size=64,
    train_split=0.7,
    val_split=0.15,
    test_split=0.15,
    random_seed=42
)

print(f"✅ Feature dim: {metadata['feature_dim']}")
print(f"   Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# STEP 3: Run hyperparameter tuning
print("\n" + "="*60)
print("STEP 3: Running Hyperparameter Tuning with Optuna")
print("="*60)

best_params, study = tune_hyperparameters(
    train_loader=train_loader,
    val_loader=val_loader,
    feature_dim=metadata['feature_dim'],
    n_trials=10
)

print("\n" + "="*60)
print("✅ HYPERPARAMETER TUNING COMPLETE")
print("="*60)
print(f"Best F1 Score: {-study.best_value:.4f}")
print("\nBest Hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print("="*60)

# ✅ Save best model config for final training
best_config_path = "best_bbtransformer_config.json"
model_keys = {
    'embed_dim', 'num_heads', 'num_layers',
    'dropout_input', 'dropout_attn', 'dropout_ffn',
    'dropout_classifier', 'dropout_temporal',
    'embed_dim_age', 'embed_dim_ext',
    'patch_size', 'patch_embed_ratio',
    'temp_attn_hidden', 'n_kv_heads'
}
final_model_config = {
    'feature_dim': metadata['feature_dim'],
    'num_classes': 1,
    **{k: v for k, v in best_params.items() if k in model_keys}
}

with open(best_config_path, 'w') as f:
    json.dump(final_model_config, f, indent=4)

print(f"\n✅ Best model config saved to: {best_config_path}")

Using device: cuda
STEP 1: Loading Data

STEP 2: Preparing Data Loaders
🧼 Cohort size: 16340 subjects
   → Cases: 7613 (46.6%)
   → Controls: 8727


[I 2025-11-15 18:47:50,683] A new study created in memory with name: bbtransformer_tuning_2025



✅ Final prevalence: 0.466
   Splits → Train: 11438, Val: 2451, Test: 2451
✅ Feature dim: 414
   Train batches: 179, Val batches: 39

STEP 3: Running Hyperparameter Tuning with Optuna


  0%|          | 0/10 [00:00<?, ?it/s]

/home/jaizor/jaizor/xtra/miniconda3/envs/Xtra/lib/python3.13/site-packages/pytorch_ranger/ranger.py:172: UserWarning: This overload of addcmul_ is deprecated:
	addcmul_(Number value, Tensor tensor1, Tensor tensor2)
Consider using one of the following signatures instead:
	addcmul_(Tensor tensor1, Tensor tensor2, *, Number value = 1) (Triggered internally at /pytorch/torch/csrc/utils/python_arg_parser.cpp:1691.)
  exp_avg_sq.mul_(beta2).addcmul_(1 - beta2, grad, grad)


[I 2025-11-15 19:13:43,342] Trial 0 finished with value: -0.9113043478260869 and parameters: {'embed_dim': 512, 'num_heads': 8, 'num_layers': 7, 'dropout_input': 0.1525940362408097, 'dropout_attn': 0.062033757292196304, 'dropout_ffn': 0.3447783107169695, 'dropout_classifier': 0.21201590646604412, 'dropout_temporal': 0.17567658225584296, 'embed_dim_age': 8, 'embed_dim_ext': 16, 'patch_size': 4, 'patch_embed_ratio': 0.75, 'temp_attn_hidden': 128, 'n_kv_heads': 4, 'lr': 1.1731776562026001e-05, 'weight_decay': 2.609494564738632e-05, 'T_0': 37}. Best is trial 0 with value: -0.9113043478260869.
[I 2025-11-15 19:22:24,851] Trial 1 finished with value: -0.8963491397398238 and parameters: {'embed_dim': 256, 'num_heads': 8, 'num_layers': 7, 'dropout_input': 0.0985319847918343, 'dropout_attn': 0.10019415943648259, 'dropout_ffn': 0.14776949942903683, 'dropout_classifier': 0.08660114974665008, 'dropout_temporal': 0.018408151700456333, 'embed_dim_age': 32, 'embed_dim_ext': 8, 'patch_size': 4, 'patch